In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')

df = pd.read_csv('../data/ai4i2020.csv')

sensor_cols = ['Air temperature [K]', 'Process temperature [K]', 
               'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

# Recreate rolling features
window_size = 10
for col in sensor_cols:
    df[f'{col}_roll_mean'] = df[col].rolling(window=window_size).mean()
    df[f'{col}_roll_std']  = df[col].rolling(window=window_size).std()
    df[f'{col}_roll_var']  = df[col].rolling(window=window_size).var()

df = df.dropna().reset_index(drop=True)
print(f"✅ Base data ready | Shape: {df.shape}")

In [ ]:
# 1. Temperature Delta (process - air) — thermal stress indicator
df['temp_delta'] = (df['Process temperature [K]'] 
                    - df['Air temperature [K]'])

# 2. Power = Torque × Rotational Speed (physics formula)
df['power'] = df['Torque [Nm]'] * df['Rotational speed [rpm]']

# 3. Tool Wear Rate — how fast is wear accumulating?
df['tool_wear_rate'] = (df['Tool wear [min]'] 
                        / (df['Rotational speed [rpm]'] + 1))

# 4. Torque per RPM — mechanical efficiency
df['torque_per_rpm'] = (df['Torque [Nm]'] 
                        / (df['Rotational speed [rpm]'] + 1))

# 5. Temp × Tool Wear interaction
df['temp_wear_interaction'] = (df['Process temperature [K]'] 
                               * df['Tool wear [min]'])

print("✅ Domain features created!")
print("\nNew features:")

new_features = [
    'temp_delta',
    'power',
    'tool_wear_rate',
    'torque_per_rpm',
    'temp_wear_interaction'
]

for f in new_features:
    print(f" → {f}: mean={df[f].mean():.3f}, std={df[f].std():.3f}")

In [ ]:
# Apply rolling window on domain features too
for col in new_features:
    df[f'{col}_roll_mean'] = df[col].rolling(window=10).mean()
    df[f'{col}_roll_std']  = df[col].rolling(window=10).std()

df = df.dropna().reset_index(drop=True)

print(f"✅ Rolling features on domain cols added | Shape: {df.shape}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(new_features):

    axes[i].hist(
        df[df['Machine failure'] == 0][col],
        bins=50,
        alpha=0.6,
        color='steelblue',
        label='No Failure'
    )

    axes[i].hist(
        df[df['Machine failure'] == 1][col],
        bins=50,
        alpha=0.8,
        color='red',
        label='Failure'
    )

    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

axes[-1].axis('off')

plt.suptitle(
    'Engineered Features: Failure vs No Failure',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()

plt.savefig('../src/engineered_features_dist.png')

plt.show()

print("✅ Plot saved!")

In [ ]:
# All numeric features correlation with failure
all_features = (
    sensor_cols
    + new_features
    + [c for c in df.columns if 'roll' in c]
)

corr_with_target = (
    df[all_features + ['Machine failure']]
    .corr()['Machine failure']
    .drop('Machine failure')
    .sort_values(key=abs, ascending=False)
)

print("=== Top 15 Features Correlated with Failure ===")
print(corr_with_target.head(15).to_string())

# Plot
plt.figure(figsize=(10, 8))

corr_with_target.head(15).plot(
    kind='barh',
    color='coral',
    edgecolor='black'
)

plt.title(
    'Top 15 Features — Correlation with Machine Failure',
    fontsize=13,
    fontweight='bold'
)

plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()

plt.savefig('../src/feature_correlation_target.png')

plt.show()

print("✅ Correlation chart saved!")

In [ ]:
# 'Type' column has L, M, H — encode it

print("Type column values:")
print(df['Type'].unique())

df['type_encoded'] = df['Type'].map({
    'L': 0,
    'M': 1,
    'H': 2
})

print("\n✅ Type encoded:")
print(df['type_encoded'].value_counts())

In [ ]:
# Drop ID cols and raw failure subtypes
# (we predict only 'Machine failure')

cols_to_drop = [
    'UDI',
    'Product ID',
    'Type',
    'TWF',
    'HDF',
    'PWF',
    'OSF',
    'RNF'
]

df_clean = df.drop(columns=cols_to_drop)

print(f"✅ Cleaned dataset shape: {df_clean.shape}")

print(f"\nFinal columns ({len(df_clean.columns)}):")

for col in df_clean.columns:
    print(f" → {col}")

In [ ]:
from collections import Counter

target_counts = Counter(
    df_clean['Machine failure']
)

print("=== Final Class Distribution ===")

print(f"No Failure (0): {target_counts[0]}")
print(f"Failure (1): {target_counts[1]}")

print(
    f"Imbalance Ratio: "
    f"{target_counts[0]/target_counts[1]:.1f}:1"
)

print(
    "\n⚠️ This confirms "
    "we NEED SMOTE later!"
)

# Pie chart
plt.figure(figsize=(6, 6))

plt.pie(
    [target_counts[0], target_counts[1]],
    labels=['No Failure', 'Failure'],
    colors=['steelblue', 'red'],
    autopct='%1.2f%%',
    startangle=90,
    explode=(0, 0.1)
)

plt.title(
    'Class Imbalance Visualization',
    fontsize=13,
    fontweight='bold'
)

plt.tight_layout()

plt.savefig('../src/class_imbalance.png')

plt.show()

In [ ]:
feature_cols = [
    c for c in df_clean.columns
    if c != 'Machine failure'
]

target_col = 'Machine failure'

print(
    f"✅ Total features ready for modeling: "
    f"{len(feature_cols)}"
)

print(
    f"✅ Target column: "
    f"{target_col}"
)

# Save feature list
with open('../src/week1_final_features.txt', 'w') as f:
    f.write(
        f"Total Features: "
        f"{len(feature_cols)}\n\n"
    )

    for feat in feature_cols:
        f.write(feat + '\n')

# Save processed dataframe info
with open('../src/week1_summary.txt', 'w') as f:
    f.write("Week 1 Final Dataset Summary\n")
    f.write("============================\n")
    f.write(f"Shape: {df_clean.shape}\n")
    f.write(f"Features: {len(feature_cols)}\n")
    f.write(
        f"Failure Rate: "
        f"{df_clean['Machine failure'].mean()*100:.2f}%\n"
    )
    f.write(
        f"Imbalance Ratio: "
        f"{target_counts[0]/target_counts[1]:.1f}:1\n"
    )

print("✅ Week 1 summary saved to src/")